# Этап 3: Предобработка данных

In [1]:
from datasets import load_dataset
from pathlib import Path
import os
from tqdm.notebook import tqdm

In [2]:
RAW_DATA_PATH = Path('../../data/raw/parquet/')
PROCESSED_DATA_PATH = Path('../../data/processed/parquet/')

Path(PROCESSED_DATA_PATH).mkdir(parents=True, exist_ok=True)

In [3]:
def process_batch(batch):
    from bs4 import BeautifulSoup
    import re
    from markdownify import markdownify as md
    
    processed_htmls = []
    
    for html_text in batch['html']:
        # 1. Валидация
        if not html_text or "Conversion to HTML had a Fatal error" in html_text:
            processed_htmls.append(None)
            continue
            
        soup = BeautifulSoup(html_text, 'html.parser')
        article = soup.find('article') or soup.find(class_='ltx_page_content')
        
        if not article:
            processed_htmls.append(None)
            continue

        # 2. Очистка мусорных блоков
        garbage_classes = [
            'ltx_bibliography', 'ltx_authors', 'ltx_role_footnotetext', 
            'ltx_navigation', 'ltx_page_footer', 'ltx_is_toc', 
            'mobile-nav', 'header', 'ltx_pagination', 'ltx_ERROR'
        ]
        
        for cls in garbage_classes:
            for tag in article.find_all(class_=cls):
                tag.decompose()
        
        # Удаляем Acknowledgements
        for header in article.find_all(['h2', 'h3'], string=re.compile(r'^\s*(Acknowledgements?|References)\s*$', re.IGNORECASE)):
            section = header.find_parent('section')
            if section:
                section.decompose()

        # 3. Удаление Base64 и технического мусора
        # Удаляем любые картинки или ссылки, закодированные прямо в HTML
        for tag in article.find_all(['img', 'a'], href=re.compile(r'^data:'), src=re.compile(r'^data:')):
            tag.decompose()

        # 4. Обработка ссылок
        for a in article.find_all('a'):
            if a.get_text(strip=True):
                a.unwrap() # .unwrap() удаляет тег, но оставляет содержимое в дереве
            else:
                a.decompose()

        # 5. Обработка картинок
        # Оставляем только значимые подписи, удаляем сами изображения
        for img in article.find_all('img'):
            alt = img.get('alt', '')
            if 'math' in str(img.get('class', [])) or len(alt) > 5:
                img.replace_with(f" [Figure: {alt}] ")
            else:
                img.decompose()

        # 6. Обработка математики (Latex)
        math_registry = {}
        for i, math in enumerate(article.find_all(class_='ltx_Math')):
            latex = math.get('alttext', '')
            if latex:
                # Используем уникальный хеш или ID, чтобы точно не пересечься с текстом
                placeholder = f"__MATH_{i}__" 
                math_registry[placeholder] = f"${latex}$"
                math.replace_with(f" {placeholder} ")

        # 7. Конвертация в Markdown
        # strip=['a', 'img'] - не трогаем уже обработанные теги
        markdown_text = md(str(article), heading_style="ATX", strip=['a', 'img'])

        # 8. Возврат математики и финальная зачистка
        for placeholder, original_latex in math_registry.items():
            markdown_text = markdown_text.replace(placeholder, original_latex)
        
        # Убираем лишние переносы строк, которые образуются после удаления блоков
        markdown_text = re.sub(r'\n\s*\n', '\n\n', markdown_text).strip()
            
        processed_htmls.append(markdown_text)

    batch['html'] = processed_htmls
    return batch

In [4]:
ds = load_dataset("parquet", data_files=str(RAW_DATA_PATH / "*.parquet"))

ds_processed = ds.map(
    process_batch,
    batched=True,
    batch_size=10,
    num_proc=os.cpu_count()
)

ds_processed = ds_processed.rename_column('html', 'md')
ds_processed['train'].to_parquet(PROCESSED_DATA_PATH / "processed_data.parquet")

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map (num_proc=12):   0%|          | 0/16345 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

812283092